# 10-K NLP Feature Construction
Three parallel tracks → merged panel → cross-sectional analysis

| Track | Source section | Method | Output |
|---|---|---|---|
| 1 | item_1A, item_7 | Loughran-McDonald sentiment | tone scores per (cik, year) |
| 2 | item_1 + item_1A + item_7 | TF-IDF cosine similarity YoY | text-change score per (cik, year) |
| 3 | item_1, item_7 | Strategy-category regex counts | shift scores per (cik, year) |

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 1. Configuration

In [ ]:
import os

CONFIG = {
    # Output of text_preprocessing.ipynb (1.8 GB, single file)
    'preprocessed_path': '/content/drive/MyDrive/FML_project_4/text_preprocessed.parquet',

    # (cik, year) whitelist from coverage_diagnostic.ipynb — survivorship bias filter
    'usable_pairs_path': '/content/drive/MyDrive/FML_project_4/usable_pairs.parquet',

    # Loughran-McDonald master dictionary
    'lm_dict_path': '/content/drive/MyDrive/FML_project_4/Loughran-McDonald_MasterDictionary_1993-2024.csv',

    # Strategy-category dictionary
    'strategy_dict_path': '/content/drive/MyDrive/FML_project_4/dictionary.csv',

    # Returns panel
    'lazy_prices_path': '/content/drive/MyDrive/FML_project_4/lazy_prices_dataset.csv',

    # Output folder (intermediate + final parquets)
    'output_folder': '/content/drive/MyDrive/FML_project_4',

    'start_year': 2004,
    'end_year':   2025,
}
print('Config OK')

## 2. Dependencies

In [ ]:
import subprocess
subprocess.run(
    ['pip', 'install', '-q', 'pyarrow', 'scikit-learn', 'tqdm'],
    check=True,
)
print('Dependencies ready')

## 3. Load preprocessed filings

In [ ]:
import pandas as pd
import numpy as np
import re
import gc
from tqdm.auto import tqdm

# ── Load preprocessed text (single parquet, no glob) ──
print('Loading text_preprocessed.parquet ...')
filings = pd.read_parquet(CONFIG['preprocessed_path'])
filings['cik']  = filings['cik'].astype(str).str.strip().str.lstrip('0')
filings['year'] = filings['year'].astype('Int64')
print(f'  {len(filings):,} rows, {filings["cik"].nunique():,} CIKs')

# ── Filter to usable (cik, year) pairs — removes survivorship bias ──
usable = pd.read_parquet(CONFIG['usable_pairs_path'])
usable['cik']  = usable['cik'].astype(str).str.strip().str.lstrip('0')
usable['year'] = usable['year'].astype('Int64')

filings = filings.merge(usable, on=['cik', 'year'], how='inner').reset_index(drop=True)
print(f'  After usable_pairs filter: {len(filings):,} rows, {filings["cik"].nunique():,} CIKs')

# Fill any missing text columns
CLEAN_COLS   = ['item_1_clean',   'item_1A_clean',   'item_7_clean']
STEMMED_COLS = ['item_1_stemmed', 'item_1A_stemmed', 'item_7_stemmed']
for col in CLEAN_COLS + STEMMED_COLS:
    if col not in filings.columns:
        filings[col] = ''
    filings[col] = filings[col].fillna('').astype(str)

filings.head(3)

## 4. Track 1 — Loughran-McDonald Sentiment Scoring

Counts tone words in `item_1A_clean` (risk factors) and `item_7_clean` (MD&A).

**Speed**: uses a reverse lookup dict (word → categories) and a `Counter` per document,
so each token is looked up once instead of once per category. ~20x faster than the
naive nested-loop approach.

**Resume**: if `lm_features.parquet` already exists on Drive, this track is skipped.

In [ ]:
LM_PATH = os.path.join(CONFIG['output_folder'], 'lm_features.parquet')

LM_CATEGORIES = ['Negative', 'Positive', 'Uncertainty', 'Litigious',
                 'Strong_Modal', 'Weak_Modal', 'Constraining']

lm_raw = pd.read_csv(CONFIG['lm_dict_path'])
lm_raw['Word'] = lm_raw['Word'].str.upper()

lm_sets: dict[str, frozenset[str]] = {
    cat: frozenset(lm_raw.loc[lm_raw[cat] != 0, 'Word'])
    for cat in LM_CATEGORIES
}

# Reverse lookup: word → list of categories it belongs to
# Means each token is dict-looked-up once, not 7 times
_lm_word_cats: dict[str, list[str]] = {}
for cat, words in lm_sets.items():
    for w in words:
        _lm_word_cats.setdefault(w, []).append(cat)

_WORD_RE = re.compile(r'[A-Z]+')

print(f'LM dictionary: {len(lm_raw):,} words, {len(_lm_word_cats):,} unique scored words')
for cat, s in lm_sets.items():
    print(f'  {cat}: {len(s):,}')

In [ ]:
from collections import Counter

if os.path.exists(LM_PATH):
    print(f'lm_features.parquet already exists — skipping Track 1.')
    sentiment_df = pd.read_parquet(LM_PATH)
else:
    def lm_scores(text: str) -> dict[str, float]:
        tokens = _WORD_RE.findall(text.upper())
        n = len(tokens)
        if n == 0:
            return {cat: 0.0 for cat in LM_CATEGORIES}
        counts: dict[str, int] = {cat: 0 for cat in LM_CATEGORIES}
        token_counts = Counter(tokens)
        # Iterate over unique tokens only, weighted by frequency
        for tok, freq in token_counts.items():
            for cat in _lm_word_cats.get(tok, []):
                counts[cat] += freq
        return {cat: counts[cat] / n for cat in LM_CATEGORIES}

    records: list[dict] = []
    for _, row in tqdm(filings.iterrows(), total=len(filings), desc='LM scoring'):
        base = {'cik': row['cik'], 'year': row['year']}
        risk = lm_scores(row['item_1A_clean'])
        mda  = lm_scores(row['item_7_clean'])
        records.append({
            **base,
            **{f'risk_{k.lower()}': v for k, v in risk.items()},
            **{f'mda_{k.lower()}':  v for k, v in mda.items()},
        })

    sentiment_df = pd.DataFrame(records)
    sentiment_df.to_parquet(LM_PATH, index=False, compression='snappy')
    print(f'Saved lm_features.parquet  {len(sentiment_df):,} rows')

sentiment_df.head(3)

## 5. Track 2 — TF-IDF Cosine Similarity (Lazy Prices)

Year-over-year text change per company using **stemmed** text (already computed in
`text_preprocessing.ipynb`). Score near 0 = nearly identical filing ("lazy"); near 1 = large rewrite.

**Resume**: skipped if `tfidf_features.parquet` already exists.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

TFIDF_PATH = os.path.join(CONFIG['output_folder'], 'tfidf_features.parquet')

if os.path.exists(TFIDF_PATH):
    print('tfidf_features.parquet already exists — skipping Track 2.')
    similarity_df = pd.read_parquet(TFIDF_PATH)
else:
    # Combine stemmed sections into one document per filing
    filings['_stemmed_full'] = (
        filings['item_1_stemmed'] + ' ' +
        filings['item_1A_stemmed'] + ' ' +
        filings['item_7_stemmed']
    )

    records: list[dict] = []
    for cik, grp in tqdm(filings.groupby('cik'), desc='TF-IDF cosine sim'):
        grp = grp.sort_values('year').reset_index(drop=True)
        if len(grp) < 2:
            continue
        texts = grp['_stemmed_full'].tolist()
        years = grp['year'].tolist()
        try:
            mat = TfidfVectorizer(
                max_features=5000,
                sublinear_tf=True,
                min_df=1,
            ).fit_transform(texts)
        except ValueError:
            continue
        for i in range(1, len(grp)):
            sim = float(cosine_similarity(mat[i], mat[i - 1])[0, 0])
            records.append({
                'cik':         cik,
                'year':        years[i],
                'text_sim':    round(sim, 6),
                'text_change': round(1.0 - sim, 6),
            })

    filings.drop(columns=['_stemmed_full'], inplace=True)
    similarity_df = pd.DataFrame(records)
    similarity_df.to_parquet(TFIDF_PATH, index=False, compression='snappy')
    print(f'Saved tfidf_features.parquet  {len(similarity_df):,} rows')

similarity_df.head(3)

## 6. Track 3 — Strategy Shift (Regex Dictionary)

Counts regex-matched strategy category mentions in `item_1_clean` and `item_7_clean`,
normalised by word count. Year-over-year diff per company = strategy shift signal.

**Resume**: processes CIKs in batches of 100. Already-completed batches are skipped.

In [ ]:
# import pyarrow as pa
# import pyarrow.parquet as pq
# import glob as _glob

# STRAT_PATH     = os.path.join(CONFIG['output_folder'], 'strategy_features.parquet')
# STRAT_TEMP_DIR = '/content/temp_strategy'
# os.makedirs(STRAT_TEMP_DIR, exist_ok=True)

# strat_dict = pd.read_csv(CONFIG['strategy_dict_path'])
# print(f'Strategy dictionary: {len(strat_dict)} terms, {strat_dict["Category"].nunique()} categories')

# # One compiled OR-pattern per category
# category_patterns: dict[str, re.Pattern] = {}
# for cat, grp in strat_dict.groupby('Category'):
#     patterns = grp['Suggested_Regex'].dropna().tolist()
#     combined = '|'.join(f'(?:{p})' for p in patterns if p)
#     if combined:
#         try:
#             category_patterns[cat] = re.compile(combined, re.IGNORECASE)
#         except re.error as exc:
#             print(f'  Bad regex for {cat}: {exc}')
# print(f'Compiled {len(category_patterns)} patterns')

In [ ]:
# STRAT_BATCH = 100   # CIKs per checkpoint batch

# if os.path.exists(STRAT_PATH):
#     print('strategy_features.parquet already exists — skipping Track 3.')
#     strategy_df = pd.read_parquet(STRAT_PATH)
# else:
#     def strategy_scores(text: str) -> dict[str, float]:
#         words = len(text.split())
#         if words == 0:
#             return {cat: 0.0 for cat in category_patterns}
#         return {
#             cat: len(pat.findall(text)) / words
#             for cat, pat in category_patterns.items()
#         }

#     all_ciks   = filings['cik'].unique().tolist()
#     cik_batches = [all_ciks[i : i + STRAT_BATCH] for i in range(0, len(all_ciks), STRAT_BATCH)]

#     # Resume: find completed batch temp files
#     done: set[int] = {
#         int(os.path.basename(f)[6:10])
#         for f in _glob.glob(f'{STRAT_TEMP_DIR}/strat_*.parquet')
#     }
#     print(f'{len(cik_batches)} batches — {len(done)} already done — {len(cik_batches) - len(done)} remaining')

#     for b_num, batch_ciks in enumerate(tqdm(cik_batches, desc='Strategy batches')):
#         tmp = f'{STRAT_TEMP_DIR}/strat_{b_num:04d}.parquet'
#         if os.path.exists(tmp):
#             continue

#         subset = filings[filings['cik'].isin(batch_ciks)].copy()
#         rows: list[dict] = []
#         for _, row in subset.iterrows():
#             base = {'cik': row['cik'], 'year': row['year']}
#             biz  = strategy_scores(row['item_1_clean'])
#             mda  = strategy_scores(row['item_7_clean'])
#             rows.append({
#                 **base,
#                 **{f'biz_{c.lower().replace(" ","_").replace("&","and")}': v for c, v in biz.items()},
#                 **{f'mda_{c.lower().replace(" ","_").replace("&","and")}': v for c, v in mda.items()},
#             })

#         pd.DataFrame(rows).to_parquet(tmp, index=False, compression='snappy')
#         del subset, rows; gc.collect()

#     # Merge temp batches
#     print('Merging strategy batches ...')
#     raw_files = sorted(_glob.glob(f'{STRAT_TEMP_DIR}/strat_*.parquet'))
#     raw_files = [f for f in raw_files if os.path.getsize(f) > 0]
#     strategy_raw = pd.concat([pd.read_parquet(f) for f in raw_files], ignore_index=True)

#     # Year-over-year diff per (cik, category)
#     strategy_raw = strategy_raw.sort_values(['cik', 'year'])
#     score_cols   = [c for c in strategy_raw.columns if c not in ('cik', 'year')]
#     strategy_df  = strategy_raw.copy()
#     strategy_df[score_cols] = strategy_raw.groupby('cik')[score_cols].diff()
#     strategy_df  = strategy_df.dropna(subset=score_cols, how='all')

#     strategy_df.to_parquet(STRAT_PATH, index=False, compression='snappy')
#     print(f'Saved strategy_features.parquet  {len(strategy_df):,} rows')

# strategy_df.head(3)

## 7. Merge all NLP features → `nlp_features.parquet`

In [ ]:
NLP_PATH = os.path.join(CONFIG['output_folder'], 'nlp_features.parquet')

nlp_features = (
    sentiment_df
    .merge(similarity_df, on=['cik', 'year'], how='outer')
    # .merge(strategy_df,   on=['cik', 'year'], how='outer')
)
nlp_features['cik']  = nlp_features['cik'].astype(str).str.strip().str.lstrip('0')
nlp_features['year'] = nlp_features['year'].astype('Int64')

nlp_features.to_parquet(NLP_PATH, index=False, compression='snappy')
print(f'nlp_features.parquet: {nlp_features.shape}')
print(f'Columns: {list(nlp_features.columns)}')
nlp_features.head(3)

## 8. Join with return data (lazy_prices_dataset)

Merge NLP features with the panel of monthly returns and accounting variables.
The filing date is matched to the return month that follows the filing.

In [ ]:
PANEL_PATH = os.path.join(CONFIG['output_folder'], 'analysis_panel.parquet')

returns = pd.read_csv(CONFIG['lazy_prices_path'])
returns['fdate']        = pd.to_datetime(returns['fdate'])
returns['return_month'] = pd.to_datetime(returns['return_month'])
returns['cik']          = returns['cik'].astype(str).str.strip().str.lstrip('0')
returns['year']         = returns['fdate'].dt.year.astype('Int64')

ret_key = (
    returns.sort_values('fdate')
           .drop_duplicates(subset=['cik', 'year'], keep='first')
    [['cik', 'year', 'accession', 'fdate', 'gvkey', 'permno', 'datadate', 'at', 'sale', 'ni']]
)
ret_panel = returns[['cik', 'year', 'return_month', 'monthly_return', 'permno']].copy()

# Merge NLP signals onto filing keys, then re-attach monthly rows
panel = ret_key.merge(nlp_features, on=['cik', 'year'], how='inner')
panel = ret_panel.merge(panel.drop(columns=['permno']), on=['cik', 'year'], how='inner')

panel.to_parquet(PANEL_PATH, index=False, compression='snappy')
print(f'analysis_panel.parquet: {panel.shape}')
print(f'Unique companies : {panel["cik"].nunique():,}')
print(f'Unique (cik, year): {panel.drop_duplicates(["cik","year"]).shape[0]:,}')
panel.head(3)

## 9. Portfolio sort preview

Quintile-sort companies each year on a chosen NLP signal and compute average next-month return
per quintile. A monotonic pattern suggests the signal carries return-predictive information.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Pick a signal to inspect ──────────────────────────────────────────────────
SIGNAL = 'text_change'   # ← change to any NLP column to explore

if SIGNAL not in panel.columns:
    print(f'Signal "{SIGNAL}" not found. Available signals:')
    print([c for c in panel.columns if c not in ('cik','year','permno','return_month',
                                                   'monthly_return','accession','fdate',
                                                   'gvkey','datadate','at','sale','ni')])
else:
    # One row per (cik, year): take next-12-month buy-and-hold return
    bahr = (
        panel.groupby(['cik', 'year'])
             .apply(lambda g: (1 + g['monthly_return'].fillna(0)).prod() - 1)
             .reset_index(name='annual_return')
    )

    signal_df = (
        panel.drop_duplicates(['cik', 'year'])[['cik', 'year', SIGNAL]]
             .merge(bahr, on=['cik', 'year'])
             .dropna(subset=[SIGNAL, 'annual_return'])
    )

    # Quintile sort within each year
    signal_df['quintile'] = (
        signal_df.groupby('year')[SIGNAL]
                 .transform(lambda x: pd.qcut(x, 5, labels=False, duplicates='drop') + 1)
    )

    q_returns = (
        signal_df.groupby('quintile')['annual_return']
                 .agg(['mean', 'count'])
                 .reset_index()
    )
    q_returns.columns = ['Quintile', 'Mean Annual Return', 'N']
    q_returns['Mean Annual Return (%)'] = (q_returns['Mean Annual Return'] * 100).round(2)

    # Long-short return (Q5 - Q1)
    ls = q_returns.loc[q_returns['Quintile'] == 5, 'Mean Annual Return (%)'].values[0] -          q_returns.loc[q_returns['Quintile'] == 1, 'Mean Annual Return (%)'].values[0]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(q_returns['Quintile'], q_returns['Mean Annual Return (%)'],
                  color=['#d62728','#ff7f0e','#7f7f7f','#2ca02c','#1f77b4'], alpha=0.85,
                  edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'Quintile sort on: {SIGNAL} Long–short (Q5−Q1): {ls:+.2f}%', fontsize=13)
    ax.set_xlabel('Quintile (1 = lowest signal, 5 = highest)')
    ax.set_ylabel('Mean Annual Return (%)')
    ax.set_xticks([1, 2, 3, 4, 5])
    for bar, row in zip(bars, q_returns.itertuples()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{row._3:.1f}%
(n={row.N})', ha='center', va='bottom', fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_folder'], f'portfolio_sort_{SIGNAL}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(q_returns[['Quintile', 'Mean Annual Return (%)', 'N']].to_string(index=False))

## 10. Next: Fama-MacBeth regressions

With `analysis_panel.parquet` ready, the next step is a Fama-MacBeth regression:

```python
import linearmodels as lm

# Controls: log(at), log(sale), ni/at, past 12m return (momentum)
# Test whether NLP signals carry incremental return-predictive power
# after controlling for size, book-to-market, and momentum
```

Load `analysis_panel.parquet` in a separate notebook and run cross-sectional regressions
for each signal to obtain t-statistics with Newey-West standard errors.